In [0]:
from pyspark.sql.functions import current_timestamp, col, to_date

VOLUME_PATH     = "/Volumes/de_workspace26/ecommerce_badal/raw/"
CHECKPOINT_PATH = "/Volumes/de_workspace26/ecommerce_badal/checkpoints/"

In [0]:
streaming_orders = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "streaming_schema")
  .option("header", "true")
  .option("inferSchema", "true")
  .load(VOLUME_PATH + "orders*.csv")
)

# COMMAND ----------
# Transform
streaming_transformed = (streaming_orders
  .withColumn("order_date",   to_date(col("order_date"), "yyyy-MM-dd"))
  .withColumn("revenue",      col("quantity") * col("unit_price"))
  .withColumn("_ingested_at", current_timestamp())
  .withWatermark("order_date", "10 minutes")
)

In [0]:
# COMMAND ----------
# 5c. Write stream to gold.live_orders with checkpoint
stream_query = (streaming_orders
  .withColumn('quantity', col('quantity').cast('double'))
  .withColumn('unit_price', col('unit_price').cast('double'))
  .withColumn("order_date",   to_date(col("order_date"), "yyyy-MM-dd").cast("timestamp"))
  .withColumn("revenue",      col("quantity") * col("unit_price"))
  .withColumn("_ingested_at", current_timestamp())
  .withWatermark("order_date", "10 minutes")
  .writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", CHECKPOINT_PATH + "live_orders")
  .trigger(availableNow=True)       # processes all available data then stops
  .table("de_workspace26.ecommerce_badal.gold_live_orders")
)

In [0]:
# COMMAND ----------
# MAGIC %md ## ─────────────────────────────────────────
# MAGIC ## TASK 5 — Structured Streaming (Module 8)
# MAGIC ## ─────────────────────────────────────────

# COMMAND ----------
from pyspark.sql.functions import current_timestamp, col, to_date

VOLUME_PATH     = "/Volumes/de_workspace26/ecommerce_badal/raw/"
CHECKPOINT_PATH = "/Volumes/de_workspace26/ecommerce_badal/checkpoints/"

# COMMAND ----------
# MAGIC %md ### Step 1 — Configure Auto Loader in Streaming Mode

# COMMAND ----------
streaming_orders = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", CHECKPOINT_PATH + "streaming_schema")
  .option("header", "true")
  .option("inferSchema", "true")
  .load(VOLUME_PATH + "orders*.csv")
)

print("✅ Streaming DataFrame created")
print(f"Is Streaming: {streaming_orders.isStreaming}")   # should print True

# COMMAND ----------
# MAGIC %md ### Step 2 — Add withWatermark on order_date

# COMMAND ----------
streaming_transformed = (streaming_orders
  .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
  .withColumn("revenue",  (col("quantity").cast("float")) * (col("unit_price").cast("float")))
  .withColumn("_ingested_at", current_timestamp())
  .withColumn("order_timestamp", col("order_date").cast("timestamp")).withWatermark("order_timestamp", "10 minutes")   # 10-minute threshold
)

print("✅ Watermark added on order_date with 10-minute threshold")

# COMMAND ----------
# MAGIC %md ### Step 3 — Write Stream to gold_live_orders

# COMMAND ----------
stream_query = (streaming_transformed
  .writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", CHECKPOINT_PATH + "live_orders")   # checkpoint path
  .trigger(availableNow=True)
  .table("de_workspace26.ecommerce_badal.gold_live_orders")
)

# Wait for stream to finish
stream_query.awaitTermination()
print("✅ Stream written to gold_live_orders")

# COMMAND ----------
# Check row count BEFORE restart
count_before = spark.table("de_workspace26.ecommerce_badal.gold_live_orders").count()
print(f"Row count BEFORE restart: {count_before}")
spark.table("de_workspace26.ecommerce_badal.gold_live_orders").display()

# COMMAND ----------
# MAGIC %md ### Step 4 — Stop and Restart the Stream

# COMMAND ----------
# Stop all active streams
for s in spark.streams.active:
    s.stop()
    print(f"🛑 Stopped stream: {s.id}")

print("✅ All streams stopped")

# COMMAND ----------
# Restart the SAME stream with SAME checkpoint path
stream_query_restart = (streaming_transformed
  .writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", CHECKPOINT_PATH + "live_orders")   # same checkpoint = no duplicates
  .trigger(availableNow=True)
  .table("de_workspace26.ecommerce_badal.gold_live_orders")
)

stream_query_restart.awaitTermination()
print("✅ Stream restarted successfully")

# COMMAND ----------
# Check row count AFTER restart
count_after = spark.table("de_workspace26.ecommerce_badal.gold_live_orders").count()
print(f"Row count BEFORE restart: {count_before}")
print(f"Row count AFTER restart:  {count_after}")

# COMMAND ----------
# Verify no duplicates
if count_before == count_after:
    print("✅ No duplicate rows — checkpoint is working correctly")
    print(f"   Same {count_after} rows before and after restart")
else:
    print(f"❌ Duplicate rows detected!")
    print(f"   Before: {count_before} | After: {count_after} | Difference: {count_after - count_before}")
    print("   Fix: Delete checkpoint and re-run")
    print(f"   dbutils.fs.rm('{CHECKPOINT_PATH}live_orders', recurse=True)")

✅ Streaming DataFrame created
Is Streaming: True
✅ Watermark added on order_date with 10-minute threshold
✅ Stream written to gold_live_orders
Row count BEFORE restart: 235


order_id,customer_id,product_id,order_date,quantity,unit_price,status,region,_rescued_data,revenue,_ingested_at
ORD0031,CUST12,PROD07,2024-04-15T00:00:00Z,4.0,34999.0,delivered,North,null,139996.0,2026-04-08T10:14:43.049Z
ORD0185,CUST03,PROD08,2024-06-28T00:00:00Z,3.0,699.0,delivered,North,null,2097.0,2026-04-08T10:14:43.049Z
ORD0060,CUST14,PROD04,2024-03-09T00:00:00Z,2.0,3499.0,delivered,North,null,6998.0,2026-04-08T10:14:43.049Z
ORD0077,CUST16,PROD10,2024-01-19T00:00:00Z,4.0,899.0,delivered,South,null,3596.0,2026-04-08T10:14:43.049Z
ORD0159,CUST05,PROD07,2024-06-16T00:00:00Z,2.0,34999.0,placed,East,null,69998.0,2026-04-08T10:14:43.049Z
ORD0058,CUST12,PROD09,2024-04-14T00:00:00Z,5.0,499.0,delivered,South,null,2495.0,2026-04-08T10:14:43.049Z
ORD0171,CUST10,PROD04,2024-04-08T00:00:00Z,4.0,3499.0,shipped,West,null,13996.0,2026-04-08T10:14:43.049Z
ORD0002,CUST15,PROD09,2024-02-01T00:00:00Z,4.0,499.0,placed,East,null,1996.0,2026-04-08T10:14:43.049Z
ORD0065,CUST04,PROD05,2024-02-15T00:00:00Z,5.0,2199.0,cancelled,East,null,10995.0,2026-04-08T10:14:43.049Z
ORD0148,CUST08,PROD02,2024-04-28T00:00:00Z,1.0,1499.0,delivered,South,null,1499.0,2026-04-08T10:14:43.049Z


✅ All streams stopped
✅ Stream restarted successfully
Row count BEFORE restart: 235
Row count AFTER restart:  235
✅ No duplicate rows — checkpoint is working correctly
   Same 235 rows before and after restart
